In [2]:
from pyspark.sql import SparkSession

# Create a spark session (which will run spark jobs)
spark = (
    SparkSession.builder.appName("MAST30034 Tutorial 1")
    .config("spark.sql.repl.eagerEval.enabled", True) 
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/09 06:39:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
import pandas as pd
CURATED_DIR = "../data/curated/curated_transactions/part_3.parquet"

# Load the ETL curated transaction level dataset 
curated_df = pd.read_parquet(CURATED_DIR)
print(curated_df.columns.tolist())
print(curated_df.info())

['user_id', 'merchant_abn', 'dollar_value', 'order_id', 'order_datetime', 'name_consumer', 'address', 'state', 'postcode', 'gender', 'consumer_id', 'name_merchant', 'tags', 'consumer_fraud_probability', 'merchant_fraud_probability', 'consumer_is_fraud', 'merchant_is_fraud', 'POA_CODE_2021', 'SA2_CODE_2021', 'mb_count', 'poa_total', 'ratio', 'SA2_NAME_2021', 'disadvantage_score', 'disadvantage_decile', 'adv_disadv_score', 'adv_disadv_decile', 'economic_resources_score', 'economic_resources_decile', 'education_occupation_score', 'education_occupation_decile', 'usual_resident_population', 'SA2_NAME_2021_income', 'IRSAD_score', 'IRSAD_decile', 'median_personal_income_weekly', 'median_family_income_weekly', 'median_household_income_weekly', 'disadvantage_score_is_approximated', 'disadvantage_decile_is_approximated', 'adv_disadv_score_is_approximated', 'adv_disadv_decile_is_approximated', 'economic_resources_score_is_approximated', 'economic_resources_decile_is_approximated', 'education_occu

In [4]:
curated_df = curated_df[[
    "merchant_abn", "user_id", "order_id", "order_datetime", "dollar_value",
    "name_merchant", "tags",
    "consumer_is_fraud", "merchant_is_fraud",
    "consumer_fraud_probability", "merchant_fraud_probability",
    "postcode", "state", "SA2_CODE_2021",
    "adv_disadv_decile", "economic_resources_decile",
]]

In [25]:
# FEATURES PRODUCED: name_merchant, category, revenue_level, take_rate
row_per_merchant = curated_df.drop_duplicates("merchant_abn").set_index("merchant_abn")
tags = row_per_merchant["tags"]

# Using regex to pull out the values out of the data
initial_feat = pd.DataFrame({
    "name_merchant": row_per_merchant["name_merchant"],
    "category": tags.str.extract(r"^[\(\[]{1,2}\s*(.*?)\s*[\)\]]\s*,\s*[\(\[]\s*[a-zA-Z]\s*[\)\]]")[0].str.strip().str.lower(),
    "revenue_level": tags.str.extract(r"[\(\[]\s*([a-zA-Z])\s*[\)\]]\s*,\s*[\(\[]\s*take rate")[0].str.lower(),
    "take_rate": tags.str.extract(r"take rate:\s*([\d.]+)")[0].astype(float),
})
if initial_feat["take_rate"].max() > 1:          # stored as a percent: convert to a fraction
    initial_feat["take_rate"] = initial_feat["take_rate"] / 100


# CHECKS: all of these should look sensible
print(initial_feat.isna().sum().to_dict())                  # every count should be 0
print(initial_feat["revenue_level"].value_counts())         # single letters like a, b, c, d, e
print(initial_feat["category"].head(3).tolist())            # no leading "[" or "(" and no trailing bracket
print(initial_feat["take_rate"].describe().round(4))

{'name_merchant': 381, 'category': 381, 'revenue_level': 381, 'take_rate': 381}
revenue_level
a    1599
b    1339
c     911
d      96
e      53
Name: count, dtype: int64
['gift, card, novelty, and souvenir shops', 'florists supplies, nursery stock, and flowers', 'gift, card, novelty, and souvenir shops']
count    3998.0000
mean        0.0440
std         0.0178
min         0.0010
25%         0.0298
50%         0.0451
75%         0.0604
max         0.0700
Name: take_rate, dtype: float64


In [27]:
# Checking if the missing merchant data (381) is an issue with the source file 
no_info = curated_df["tags"].isna()

merchants_raw = pd.read_parquet("../data/tables/part_1/tbl_merchants.parquet")      # adjust the path
missing_abns = curated_df.loc[no_info, "merchant_abn"].unique()
print(merchants_raw.index.isin(missing_abns).sum(), "of", len(missing_abns), "are in the merchants file")

0 of 381 are in the merchants file


In [28]:
# Merchants that appear in transactions but not in the merchants file (381 of them) will be flagged to have no tags (unknown)
initial_feat["has_merchant_info"] = tags.notna()
initial_feat["category"] = initial_feat["category"].fillna("unknown")

In [ ]:
# Grouping trasaction level data by merchants 
df_by_merch = curated_df.groupby("merchant_abn")

# FEATURES PRODUCED: total_revenue, n_transactions, avg_order_value, n_customers
feat_df = pd.DataFrame({
    "total_revenue": df_by_merch["dollar_value"].sum(),          # revenue: merchant's gross dollar sales
    "n_transactions": df_by_merch["order_id"].size(),            # sales: number of orders
    "avg_order_value": df_by_merch["dollar_value"].mean(),       # average order size
    "n_customers": df_by_merch["user_id"].nunique(),             # customer base
})

# FEATURE PRODUCED: orders_per_customer (how often an average customer buys)
feat_df ["orders_per_customer"] = feat_df ["n_transactions"] / feat_df ["n_customers"]

# FEATURE PRODUCED: avg_weekly_transactions (orders per week over the merchant's own trading span)
span_days = (df_by_merch["order_datetime"].max() - df_by_merch["order_datetime"].min()).dt.days + 1
feat_df ["avg_weekly_transactions"] = feat_df["n_transactions"] / (span_days / 7).clip(lower=1)

# FEATURES PRODUCED: repeat_rate, top_customer_share
merchant_customer = curated_df.groupby(["merchant_abn", "user_id"]).agg(
    n=("order_id", "size"), spend=("dollar_value", "sum"))
merchant_customer["is_repeat"] = merchant_customer["n"] >= 2
customer_summary = merchant_customer.groupby("merchant_abn").agg(
    repeat_rate=("is_repeat", "mean"), top_spend=("spend", "max"), tot_spend=("spend", "sum"))
feat_df["repeat_rate"] = customer_summary["repeat_rate"]
feat_df["top_customer_share"] = customer_summary["top_spend"] / customer_summary["tot_spend"]

# FEATURES PRODUCED: revenue_growth, revenue_cv, active_months
month = curated_df["order_datetime"].dt.to_period("M")
monthly_revenue = curated_df.groupby(["merchant_abn", month])["dollar_value"].sum().unstack(fill_value=0)
half = monthly_revenue.shape[1] // 2
first_half_revenue = monthly_revenue.iloc[:, :half].sum(axis=1)
second_half_revenue = monthly_revenue.iloc[:, half:].sum(axis=1)
feat_df["revenue_growth"] = (second_half_revenue - first_half_revenue) / (first_half_revenue + second_half_revenue)
feat_df["revenue_cov"] = monthly_revenue.std(axis=1) / monthly_revenue.mean(axis=1)
feat_df["active_months"] = (monthly_revenue > 0).sum(axis=1)

In [ ]:
# fraud flags are booleans; fraud_value is the dollar value of fraudulent-consumer transactions
fraud_df = curated_df[["merchant_abn", "dollar_value", "consumer_is_fraud", "merchant_is_fraud",
       "consumer_fraud_probability", "merchant_fraud_probability"]].copy()
fraud_df["fraud_value"] = fraud_df["dollar_value"] * fraud_df["consumer_is_fraud"].astype(float)
fraud_by_merch = fraud_df.groupby("merchant_abn")

# FEATURES PRODUCED: consumer_fraud_rate, fraud_value_share, merchant_fraud_rate,
#                    avg_consumer_fraud_prob, max_merchant_fraud_prob
fraud_feat = pd.DataFrame({
    "consumer_fraud_rate": fraud_by_merch ["consumer_is_fraud"].mean(),                    # share of orders from fraudulent consumers
    "fraud_value_share": fraud_by_merch ["fraud_value"].sum() / fraud_by_merch ["dollar_value"].sum(),  # share of dollars from fraud (main risk feature)
    "merchant_fraud_rate": fraud_by_merch ["merchant_is_fraud"].mean(),                    # share of orders flagged as merchant fraud
    "avg_consumer_fraud_prob": fraud_by_merch ["consumer_fraud_probability"].mean(),       # reference only
    "max_merchant_fraud_prob": fraud_by_merch ["merchant_fraud_probability"].max(),        # reference only
})
# No probability recorded means there no flagged days, so treat it as 0 (not missing)
fraud_feat = fraud_feat.fillna({"avg_consumer_fraud_prob": 0, "max_merchant_fraud_prob": 0})

In [10]:
# FEATURES PRODUCED: n_postcodes, n_states
geo_feat = pd.DataFrame({
    "n_postcodes": curated_df.groupby("merchant_abn")["postcode"].nunique(),
    "n_states": curated_df.groupby("merchant_abn")["state"].nunique(),
})

# FEATURE PRODUCED:  n_postcodes, top_postcode_share, top5_postcode_share
postcode_spend = curated_df.groupby(["merchant_abn", "postcode"])["dollar_value"].sum()

# Each postcode's share of that merchant's total sales 
postcode_share = postcode_spend / postcode_spend.groupby("merchant_abn").transform("sum")

# The geographic features, one row per merchant
geo_feat = pd.DataFrame({
    "n_postcodes": curated_df.groupby("merchant_abn")["postcode"].nunique(),   # how many postcodes it sells to
    "top_postcode_share": postcode_share.groupby("merchant_abn").max(),        # share of sales from the postcodes with the highest sales
    "top5_postcode_share": (postcode_share.sort_values(ascending=False)
                                          .groupby("merchant_abn").head(5)     # keep the top 5 highest sales postcodes for each of the 
                                          .groupby("merchant_abn").sum()),     # merchant's and add up their shares
})

# FEATURE PRODUCED: no_sa2_match: share of a merchant's orders with no SA2/SEIFA match for postcode)
geo_feat["no_sa2_match"] = curated_df["SA2_CODE_2021"].isna().groupby(curated_df["merchant_abn"]).mean()

# FEATURES PRODUCED: w_adv_disadv_decile, w_economic_resources_decile
# Produces a dollar-weighted SEIFA decile of the customers' areas
# Orders with no SEIFA value are left out, not counted as 0
for col in ["adv_disadv_decile", "economic_resources_decile"]:
    has_value = curated_df[col].notna() 
    weighted_sum = (curated_df[col] * curated_df["dollar_value"]).groupby(curated_df["merchant_abn"]).sum()
    total_weight = curated_df["dollar_value"].where(has_value).groupby(curated_df["merchant_abn"]).sum()
    geo_feat["w_" + col] = weighted_sum / total_weight



In [29]:
# FEATURE PRODUCED: expected_bnpl_revenue (what the BNPL firm earns = revenue x take rate)
feat_df = feat_df.join([initial_feat, fraud_feat, geo_feat])
feat_df["expected_bnpl_revenue"] = feat_df["total_revenue"] * feat_df["take_rate"]

assert feat_df.index.is_unique                                     # one row per merchant
assert len(feat_df) == curated_df["merchant_abn"].nunique()        # none dropped or duplicated

In [36]:
# Cut-offs for the flags: the 10th, 25th and 75th percentile of each measure across all merchants
flag_thresholds = feat_df[["n_customers", "avg_order_value", "fraud_value_share", "active_months"]].quantile([0.1, 0.25, 0.75])

# FLAGS PRODUCED: flag_small_base_big_spender, flag_large_base_high_fraud, flag_new_merchant
# Few customers (bottom 25%) but large orders (top 25%)
feat_df["flag_small_base_big_spender"] = ((feat_df["n_customers"] <= flag_thresholds.loc[0.25, "n_customers"])
                                          & (feat_df["avg_order_value"] >= flag_thresholds.loc[0.75, "avg_order_value"]))

# Many customers (top 25%) AND a high share of sales from fraud (top 25%, and above zero)
feat_df["flag_large_base_high_fraud"] = ((feat_df["n_customers"] >= flag_thresholds.loc[0.75, "n_customers"])
                                         & (feat_df["fraud_value_share"] >= flag_thresholds.loc[0.75, "fraud_value_share"])
                                         & (feat_df["fraud_value_share"] > 0))  

# Few months of trading history (bottom 10%)
feat_df["flag_new_merchant"] = feat_df["active_months"] <= flag_thresholds.loc[0.1, "active_months"]

flag_cols = [c for c in feat_df if c.startswith("flag_")]
print(feat_df[flag_cols].sum())

# Extract the flagged merchants from feat_df and checking if the columns exist in our table
flag_cols_needed = ["name_merchant", "category", "n_customers", "fraud_value_share", "total_revenue"]
show_cols = [c for c in flag_cols_needed if c in feat_df.columns]

display(feat_df[feat_df["flag_large_base_high_fraud"]]
        .sort_values("fraud_value_share", ascending=False)
        .head(10)[show_cols])

flag_small_base_big_spender    660
flag_large_base_high_fraud      33
flag_new_merchant              469
dtype: int64


,name_merchant,category,n_customers,fraud_value_share,total_revenue
merchant_abn,,,,,
76251945060,Tempor Arcu Corp.,health and beauty spas,658,0.002431,3.262391e+05
55778594682,Auctor Nunc Consulting,"digital goods: books, movies, music",1238,0.001039,6.536757e+04
49549583265,Luctus Et Incorporated,tent and awning shops,3029,0.000778,2.163493e+06
92779316513,Nec Malesuada Industries,"cable, satellite, and other pay television and...",2859,0.000694,3.000557e+05
61412665910,Tempor Inc.,"furniture, home furnishings and equipment shop...",2753,0.000679,3.874997e+05
74391255526,Vitae Semper Egestas Foundation,health and beauty spas,1640,0.000677,3.158346e+05
88358066529,Pede Cras Vulputate Consulting,health and beauty spas,1557,0.000593,8.301339e+05
34695415993,Ultrices Vivamus Rhoncus Associates,"cable, satellite, and other pay television and...",2568,0.000550,1.595209e+05
72857570096,Neque Incorporated,shoe shops,1456,0.000371,3.559468e+05


In [ ]:
print(feat_df.shape)                    
print(feat_df.index.is_unique)                    # Checking if the merchants are unique: True 

missing_counts = feat_df.isna().sum()             # Count blanks in each column
print(missing_counts[missing_counts > 0])         # Output the columns with the missing values

feature_cols = ["total_revenue", "expected_bnpl_revenue", "avg_weekly_transactions", "repeat_rate",
                "n_postcodes", "top_postcode_share", "fraud_value_share", "merchant_fraud_rate",
                "revenue_growth", "revenue_cov", "n_customers", "top_customer_share"]

display(feat_df[feature_cols].describe().T[["count", "min", "50%", "mean", "max"]].round(3))

(4379, 31)
True
name_merchant            381
revenue_level            381
take_rate                381
expected_bnpl_revenue    381
dtype: int64


,count,min,50%,mean,max
total_revenue,4379.0,785.702,46002.258,171308.379,3192782.749
expected_bnpl_revenue,3998.0,4.646,1667.112,7661.542,210355.340
avg_weekly_transactions,4379.0,0.081,4.641,39.248,3506.772
repeat_rate,4379.0,0.000,0.000,0.019,0.914
n_postcodes,4379.0,1.000,118.000,451.807,3165.000
top_postcode_share,4379.0,0.001,0.031,0.105,1.000
fraud_value_share,4379.0,0.000,0.000,0.002,1.000
merchant_fraud_rate,4379.0,0.000,0.000,0.000,0.059
revenue_growth,4379.0,-1.000,0.339,0.337,1.000
revenue_cov,4379.0,0.242,0.546,0.725,2.646


In [38]:
# data output directory is
OUTPUT_DIR = '../data/'

# save the cleaned POA <-> SA2 mapping df to be reused in the next step 
feat_df.reset_index().to_parquet(f"{OUTPUT_DIR}/merchant_features.parquet", index=False)